# 16 — Contract Price Tracker
Tracks how the market-implied price for one specific delivery period has evolved over time as it rolled from a far-dated contract toward expiry.

Supports: single month, quarter, season (Summer/Winter), calendar year.

In [ ]:
import sys, os, warnings, re, calendar
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.colors as pc
from pathlib import Path
warnings.filterwarnings("ignore")

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / "src" / "agsi_client").exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f"Project root: {_c}"); break

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION — set one target contract here
TARGET  = "Mar 2027"      # single month

# Supported formats:
#   "Mar 2027"     → March 2027 (month name + 4-digit year)
#   "Q3 2027"      → Q3 2027 = Jul/Aug/Sep 2027
#   "Summer 25"    → Apr-Sep 2025   (2-digit year ok)
#   "Winter 25"    → Oct 2025 - Mar 2026
#   "CAL 2026"     → all 12 months of 2026

# Multiple targets for comparison chart (Cell 5)
TARGETS = ["Mar 2027", "Summer 26", "Winter 26", "CAL 2026"]
# ═══════════════════════════════════════════════════════════════════

RAW = Path("data/raw")
ttf = pd.read_csv(RAW / "ttf_curve.csv", index_col=0, parse_dates=True)
ttf.index = pd.to_datetime(ttf.index).tz_localize(None)
ttf = ttf.sort_index()

print(f"TTF curve loaded: {ttf.shape}")
print(f"  {ttf.index[0].date()} -> {ttf.index[-1].date()}")
print(f"  Columns: {list(ttf.columns[:6])} ...")
print(f"\nTarget: {TARGET}")


## 1 · Target Parser
Parse the `TARGET` string into a list of `(delivery_year, delivery_month)` tuples.

In [ ]:
# ── Month name lookup (abbr + full, case-insensitive) ─────────────────────────
_MONTH_MAP = {}
for i in range(1, 13):
    _MONTH_MAP[calendar.month_abbr[i].lower()] = i
    _MONTH_MAP[calendar.month_name[i].lower()]  = i

def parse_target(target_str):
    """
    Return list of (delivery_year, delivery_month) tuples for a contract string.

    Formats:
      "Mar 2027"    → [(2027, 3)]
      "March 2027"  → [(2027, 3)]
      "Q3 2027"     → [(2027, 7), (2027, 8), (2027, 9)]
      "Summer 25"   → [(2025, 4), ..., (2025, 9)]   Apr-Sep
      "Winter 25"   → [(2025,10), ..., (2026, 3)]   Oct-Mar
      "CAL 2026"    → [(2026, 1), ..., (2026,12)]
    """
    s = target_str.strip()

    # CAL YYYY
    m = re.match(r"^CAL\s+(\d{4})$", s, re.I)
    if m:
        yr = int(m.group(1))
        return [(yr, mo) for mo in range(1, 13)]

    # Q[1-4] YYYY
    m = re.match(r"^Q([1-4])\s+(\d{4})$", s, re.I)
    if m:
        q, yr = int(m.group(1)), int(m.group(2))
        start = (q - 1) * 3 + 1
        return [(yr, mo) for mo in range(start, start + 3)]

    # Summer YY or Summer YYYY
    m = re.match(r"^Summer\s+(\d{2,4})$", s, re.I)
    if m:
        raw = m.group(1)
        yr = int(raw) if len(raw) == 4 else 2000 + int(raw)
        return [(yr, mo) for mo in range(4, 10)]

    # Winter YY or Winter YYYY  →  Oct of yr through Mar of yr+1
    m = re.match(r"^Winter\s+(\d{2,4})$", s, re.I)
    if m:
        raw = m.group(1)
        yr = int(raw) if len(raw) == 4 else 2000 + int(raw)
        return [(yr, 10), (yr, 11), (yr, 12), (yr+1, 1), (yr+1, 2), (yr+1, 3)]

    # Month YYYY  (abbr or full)
    m = re.match(r"^(\w+)\s+(\d{4})$", s)
    if m:
        mo_str, yr = m.group(1).lower(), int(m.group(2))
        if mo_str in _MONTH_MAP:
            return [( yr, _MONTH_MAP[mo_str] )]

    raise ValueError(f"Cannot parse: {target_str!r}\n"
                     "Supported: 'Mar 2027', 'Q3 2027', 'Summer 25', 'Winter 25', 'CAL 2026'")


# ── Test the parser ────────────────────────────────────────────────────────────
delivery_months = parse_target(TARGET)
print(f"TARGET: {TARGET!r}")
print(f"Delivery months ({len(delivery_months)} total):")
for yr, mo in delivery_months:
    print(f"  {calendar.month_abbr[mo]} {yr}")


## 2 · Build Price History
For each trade date in `ttf_curve.csv`, look up the M+N column(s) for the target delivery month(s) and build a time series.

In [ ]:
def build_price_series(ttf_df, delivery_months):
    """
    For each trade date, find M+N columns for each delivery month and average them.
    months_ahead = (delivery_year - trade_year)*12 + (delivery_month - trade_month)
    Returns a pd.Series indexed by trade date.
    """
    rows = []
    for trade_dt, trade_row in ttf_df.iterrows():
        prices = []
        for (del_yr, del_mo) in delivery_months:
            months_ahead = (del_yr - trade_dt.year) * 12 + (del_mo - trade_dt.month)
            if 1 <= months_ahead <= 24:
                col = f"M{months_ahead}"
                if col in trade_row.index and not pd.isna(trade_row[col]):
                    prices.append(float(trade_row[col]))
        if prices:
            rows.append({
                "date": trade_dt,
                "price": np.mean(prices),
                "n_legs": len(prices),
            })
    if not rows:
        return pd.Series(dtype=float, name="price")
    df_out = pd.DataFrame(rows).set_index("date").sort_index()
    return df_out["price"]


price_series = build_price_series(ttf, delivery_months)

if price_series.empty:
    print(f"⚠️  No quotable dates found for {TARGET}.")
    print(f"    Delivery months span {delivery_months[0]} → {delivery_months[-1]}")
    print(f"    TTF data range: {ttf.index[0].date()} → {ttf.index[-1].date()}")
else:
    print(f"Price series for {TARGET!r}")
    print(f"  First quote : {price_series.index[0].date()}  →  {price_series.iloc[0]:.3f} EUR/MWh")
    print(f"  Latest      : {price_series.index[-1].date()} →  {price_series.iloc[-1]:.3f} EUR/MWh")
    print(f"  N obs       : {len(price_series):,} trading days")
    print(f"  Min         : {price_series.min():.3f}  ({price_series.idxmin().date()})")
    print(f"  Max         : {price_series.max():.3f}  ({price_series.idxmax().date()})")


## 3 · Price Evolution Chart
Line chart from first quotable date to latest, with annotations at first/last/min/max.

In [ ]:
if price_series.empty:
    print("⚠️  No data to chart.")
else:
    ps = price_series

    idx_min  = ps.idxmin()
    idx_max  = ps.idxmax()
    p_first  = float(ps.iloc[0])
    p_last   = float(ps.iloc[-1])
    p_min    = float(ps.min())
    p_max    = float(ps.max())

    pad = (p_max - p_min) * 0.08
    y_range = [p_min - pad, p_max + pad]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=ps.index, y=ps.values,
        mode="lines",
        line=dict(color="#2E75B6", width=1.8),
        name=TARGET,
    ))

    # Annotations: first, last, min, max
    annotations = [
        dict(x=ps.index[0],  y=p_first, text=f"First<br>{p_first:.2f}",
             showarrow=True, arrowhead=2, ax=40, ay=-30, font=dict(color="#555")),
        dict(x=ps.index[-1], y=p_last,  text=f"Latest<br>{p_last:.2f}",
             showarrow=True, arrowhead=2, ax=-45, ay=-30, font=dict(color="#E74C3C")),
        dict(x=idx_min,      y=p_min,   text=f"Min {p_min:.2f}<br>{idx_min.strftime('%b %Y')}",
             showarrow=True, arrowhead=2, ax=0,  ay=40,  font=dict(color="#27AE60")),
        dict(x=idx_max,      y=p_max,   text=f"Max {p_max:.2f}<br>{idx_max.strftime('%b %Y')}",
             showarrow=True, arrowhead=2, ax=0,  ay=-40, font=dict(color="#E74C3C")),
    ]
    # Deduplicate overlapping annotations (first == min or max)
    seen = set()
    clean_annotations = []
    for ann in annotations:
        key = (ann["x"], round(ann["y"], 1))
        if key not in seen:
            seen.add(key)
            clean_annotations.append(ann)

    fig.update_layout(
        title=f"Market-Implied Price for {TARGET} — Evolution from First Quote to Latest",
        xaxis_title="Trade Date",
        yaxis=dict(title="Price (EUR/MWh)", range=y_range),
        annotations=clean_annotations,
        height=480, template="plotly_white",
    )
    fig.show()


## 4 · Contract Statistics
First/last price, % change, realized volatility, max drawdown.

In [ ]:
if price_series.empty:
    print("⚠️  No data.")
else:
    ps = price_series.dropna()

    # Realized annualized volatility (log returns)
    log_ret = np.log(ps / ps.shift(1)).dropna()
    ann_vol = float(log_ret.std() * np.sqrt(252)) * 100   # %

    # Max drawdown (from rolling peak)
    roll_max = ps.cummax()
    drawdown = (ps - roll_max) / roll_max * 100
    max_dd   = float(drawdown.min())

    # % change first to last
    pct_chg = (ps.iloc[-1] / ps.iloc[0] - 1) * 100

    print(f"Contract Price Statistics — {TARGET}")
    print("=" * 55)
    print(f"  Delivery months  : {len(delivery_months)} "
          f"({calendar.month_abbr[delivery_months[0][1]]} {delivery_months[0][0]}"
          + (f" – {calendar.month_abbr[delivery_months[-1][1]]} {delivery_months[-1][0]}"
             if len(delivery_months) > 1 else "") + ")")
    print(f"  First quote date : {ps.index[0].date()}")
    print(f"  Last quote date  : {ps.index[-1].date()}")
    print(f"  N trading days   : {len(ps):,}")
    print(f"  First price      : {ps.iloc[0]:.3f} EUR/MWh")
    print(f"  Latest price     : {ps.iloc[-1]:.3f} EUR/MWh")
    print(f"  % change         : {pct_chg:+.1f}%")
    print(f"  Min              : {ps.min():.3f}  ({ps.idxmin().date()})")
    print(f"  Max              : {ps.max():.3f}  ({ps.idxmax().date()})")
    print(f"  Realized vol     : {ann_vol:.1f}% p.a. (annualized, log-return)")
    print(f"  Max drawdown     : {max_dd:.1f}%")


## 5 · Multi-Target Comparison
Overlay multiple contracts on one chart, normalized to % change from first observation.
Set the `TARGETS` list in Cell 0.

In [ ]:
COLORS_LIST = pc.qualitative.Set1 + pc.qualitative.Set2

fig_abs  = go.Figure()   # absolute prices
fig_norm = go.Figure()   # normalized to % change from first observation

print(f"Building price history for {len(TARGETS)} contracts...")
for i, tgt in enumerate(TARGETS):
    try:
        dm = parse_target(tgt)
    except ValueError as e:
        print(f"  ⚠️  Skipping {tgt!r}: {e}")
        continue

    ps = build_price_series(ttf, dm)
    if ps.empty:
        print(f"  ⚠️  {tgt}: no quotable dates in TTF data range")
        continue

    ps = ps.dropna()
    color = COLORS_LIST[i % len(COLORS_LIST)]
    p_first = float(ps.iloc[0])
    ps_norm = (ps / p_first - 1) * 100   # % change from first observation

    print(f"  {tgt:<18}: {len(ps):>4} obs  "
          f"{ps.index[0].date()} → {ps.index[-1].date()}  "
          f"first={p_first:.2f}  latest={ps.iloc[-1]:.2f}")

    fig_abs.add_trace(go.Scatter(
        x=ps.index, y=ps.values,
        name=tgt, mode="lines",
        line=dict(color=color, width=1.8),
    ))
    fig_norm.add_trace(go.Scatter(
        x=ps_norm.index, y=ps_norm.values,
        name=tgt, mode="lines",
        line=dict(color=color, width=1.8),
    ))

fig_abs.update_layout(
    title="Contract Price Evolution — Absolute (EUR/MWh)",
    xaxis_title="Trade Date", yaxis_title="Price (EUR/MWh)",
    height=460, template="plotly_white",
    legend=dict(orientation="h", y=-0.18),
)
fig_norm.add_hline(y=0, line_dash="dot", line_color="black", line_width=1)
fig_norm.update_layout(
    title="Contract Price Evolution — Normalized (% change from first observation)",
    xaxis_title="Trade Date", yaxis_title="% change from first quote",
    height=460, template="plotly_white",
    legend=dict(orientation="h", y=-0.18),
)
fig_abs.show()
fig_norm.show()
